## Building the Pair Stitcher

# Building the Master Stitching Pipeline

Welcome to the second unit of our course! In our previous unit, we learned how to composite and crop aligned images to create a clean, merged panorama. Now, we are going to build the engine that powers our entire panorama maker.

Our goal is to take all the separate tools we have built or learned about so far—such as finding keypoints, matching features, and estimating geometry—and combine them into a single, reliable function called `stitch_pair`. This master function will take two raw images, attempt to stitch them, handle any errors gracefully, and return both the final image and a detailed diagnostic report.

---

## Quick Recall: The Image Stitching Pipeline

Before we start coding, let us briefly recall the sequence of operations required to stitch two images together:

* **Preprocessing:** Convert images to grayscale so algorithms can process them easily.
* **Detection and Computation:** Find unique points (keypoints) in both images and describe their surrounding patterns.
* **Matching:** Pair up matching descriptions between the left and right images.
* **Homography Estimation:** Use the matched points to figure out the math (a geometric grid) needed to warp one image so that it aligns with the other.
* **Warping and Compositing:** Stretch the image using our math, overlap the two images, blend them, and crop out any unwanted black borders (as we practiced in the last unit).

We will rely on pre-built helper functions for the heavy math and geometry. This allows us to focus entirely on the integration logic: making sure the pipeline runs smoothly from start to finish.

---

## Step 1: Setting Up the Tracking Report

Let us begin writing our `stitch_pair` function. Whenever you build a complex pipeline, it is very helpful to keep track of what is happening under the hood. We will do this by creating a report dictionary.

```python
from cvkit import preprocess_for_features
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography, warp_to_canvas

def stitch_pair(left, right, method="sift", ratio=0.75, ransac_threshold=5.0):
    report = {
        "status": "failed",
        "homography_direction": "left/source -> right/target",
        "method": method,
        "ratio": ratio,
        "ransac_threshold": ransac_threshold,
        "keypoints_left": 0,
        "keypoints_right": 0,
        "matches": 0,
        "inliers": 0,
        "inlier_ratio": 0.0,
        "diagnosis": "",
    }
    
    return None, report

```

In this initial code, we define our function to accept two images (`left` and `right`) and a few settings. Inside, we create the report dictionary.

> **Note:** We set the initial "status" to "failed". This is a defensive programming practice: we assume the process will fail until it successfully completes every step. The report also holds slots to track how many keypoints we find, how many matches we make, and a space for a "diagnosis" message to help us understand what went wrong if it fails.

---

## Step 2: Executing the Pipeline Safely

Now, let us start processing the images. We will run our preprocessing and feature detection functions and immediately update our report with the results.

```python
    # Preprocess images
    left_gray = preprocess_for_features(left)
    right_gray = preprocess_for_features(right)

    # Detect features and compute descriptors
    kp1, des1 = detect_and_compute(left_gray, method=method)
    kp2, des2 = detect_and_compute(right_gray, method=method)

    # Match descriptors between the two images
    matches = match_descriptors(des1, des2, ratio=ratio)

    # Update our tracking report
    report["keypoints_left"] = len(kp1)
    report["keypoints_right"] = len(kp2)
    report["matches"] = len(matches)

```

At this point, we have matched features. However, calculating a homography (the mathematical alignment grid) requires at least four matched points. If we do not have four points, the math will crash. Let us add a safety check to stop the process early if necessary:

```python
    if len(matches) < 4:
        report["diagnosis"] = "fewer than four matches; homography cannot be estimated"
        return None, report

```

If we have enough matches, we move on to estimating the homography. Because complex geometry can sometimes fail even with enough points, we wrap this in a `try-except` block to catch any mathematical errors.

```python
    try:
        homography, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=ransac_threshold,
        )
    except ValueError as exc:
        report["diagnosis"] = f"homography failed: {exc}"
        return None, report

    # Update inlier statistics
    report["inliers"] = int(inliers.sum())
    report["inlier_ratio"] = float(inliers.mean()) if len(inliers) else 0.0

    if report["inliers"] < 4:
        report["diagnosis"] = "fewer than four inliers; alignment is not reliable"
        return None, report

```

Here, we try to calculate the homography. The `estimate_homography` function also returns inliers (the matches that are geometrically consistent within the RANSAC reprojection threshold, discarding bad matches). This `ransac_threshold` controls how much alignment error is tolerated: a smaller value requires more precise consistency, while a larger one is more lenient. We calculate the total number of inliers and check if we have fewer than four. If we do, we update the diagnosis and safely return `None` instead of a broken image.

---

## Step 3: Finishing the Stitch

If our code survives all the safety checks, we have a valid homography! Now we can warp the images and merge them using the `composite` and `crop_black` logic we learned in the previous unit.

Before we finish the function, let us define a quick helper function to give us a friendly diagnosis message for successful (or mostly successful) stitches.

```python
def diagnose_stitch(report):
    if report["matches"] < 20:
        return "too few descriptor matches; choose images with more overlap or texture"
    if report["inliers"] < 10:
        return "too few geometric inliers; inspect the inlier view"
    if report["inlier_ratio"] < 0.25:
        return "low inlier ratio; try a stricter ratio threshold or a better image pair"
    
    return "alignment probably usable; inspect the seam, exposure differences, and parallax"

```

This helper looks at the numbers in our report and provides advice. Now, we add the final lines to our `stitch_pair` function:

```python
    # Warp and composite
    warped, target_canvas = warp_to_canvas(left, right, homography)
    
    # We assume 'composite' and 'crop_black' are defined based on our previous unit
    result = crop_black(composite(warped, target_canvas))

    # Mark as success!
    report["status"] = "ok"
    report["diagnosis"] = diagnose_stitch(report)

    return result, report

```

If we reach the bottom of the function, everything has worked. We change the "status" to "ok", get our final diagnostic advice, and return the result image alongside our fully populated report.

---

## Step 4: Running the Code via the Command Line

To make our new `stitch_pair` function useful, we need a way to run it from our computer's terminal. We can do this by setting up a main execution script that takes inputs from the user.

First, we set up `argparse` to read the image filenames and settings the user types into the command line.

```python
import argparse
import cv2
from cvkit import read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--out", default="stitched_pair.jpg")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

```

Next, we pass those read images directly into our `stitch_pair` function.

```python
    result, report = stitch_pair(
        left,
        right,
        method=args.method,
        ratio=args.ratio,
        ransac_threshold=args.ransac_threshold,
    )

    # Print the report to the console
    for key, value in report.items():
        print(f"{key}: {value}")

```

If you ran this in your terminal, the printed report output would look something like this:

```text
status: ok
homography_direction: left/source -> right/target
method: sift
ratio: 0.75
ransac_threshold: 5.0
keypoints_left: 1250
keypoints_right: 1100
matches: 150
inliers: 112
inlier_ratio: 0.746
diagnosis: alignment probably usable; inspect the seam, exposure differences, and parallax

```

Finally, we handle the result. If the function returned `None`, we stop the program. If it succeeded, we save the resulting image to a file and display it on the screen.

```python
    if result is None:
        raise SystemExit("stitching failed; see diagnosis above")

    # Save and show the successful stitch
    cv2.imwrite(args.out, result)
    cv2.imshow("stitched pair", result)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

---

## Summary and Next Steps

Great job! In this lesson, we successfully wrapped our image stitching pipeline into a single, clean function. By adding step-by-step checks, a tracking report, and safe error handling, we transformed a loose collection of scripts into a robust tool. If an image pair lacks enough matching points to stitch safely, our program no longer crashes—it simply explains exactly why it could not complete the job.

Now, it is your turn to put this into action. In the upcoming practice exercises, you will head into the CodeSignal IDE to write this master function yourself, apply the error-handling logic, and test it against real image pairs.

## Diagnosing the Quality of a Stitch

Now that the lesson has guided you through building the entire pair stitcher, it is time to put one of its helper functions to work.

The diagnose_stitch function reads the values in a report dictionary and returns a short message that explains how the stitch proceeded.

Your job is to complete the four checks inside the function, in order:

    If report["matches"] is fewer than 20, return the "too few descriptor matches" message.
    If report["inliers"] is fewer than 10, return the "too few geometric inliers" message.
    If report["inlier_ratio"] is below 0.25, return the "low inlier ratio" message.
    If none of those checks trigger, return the "alignment probably usable" message.

Each TODO comment already shows the exact text to return, so match it carefully.

Perfectly implementing this helper ensures your stitcher can explain its own results, which is a great first step toward a tool people can trust.

```
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))

    if coords is None:
        return image

    x, y, w, h = cv2.boundingRect(coords)
    return image[y : y + h, x : x + w]


def diagnose_stitch(report):
    # TODO: If the "matches" value is fewer than 20, return this message:
    # "too few descriptor matches; choose images with more overlap or texture"

    # TODO: If the "inliers" value is fewer than 10, return this message:
    # "too few geometric inliers; inspect the inlier view"

    # TODO: If the "inlier_ratio" value is below 0.25, return this message:
    # "low inlier ratio; try a stricter ratio threshold or a better image pair"

    # TODO: If every check above passes, return this message:
    # "alignment probably usable; inspect the seam, exposure differences, and parallax"
    return ""
```

Here is the completed implementation for `diagnose_stitch` in `stitching.py`:

```python
import cv2
import numpy as np


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))

    if coords is None:
        return image

    x, y, w, h = cv2.boundingRect(coords)
    return image[y : y + h, x : x + w]


def diagnose_stitch(report):
    # 1. Check match count
    if report["matches"] < 20:
        return "too few descriptor matches; choose images with more overlap or texture"

    # 2. Check inlier count
    if report["inliers"] < 10:
        return "too few geometric inliers; inspect the inlier view"

    # 3. Check inlier ratio
    if report["inlier_ratio"] < 0.25:
        return "low inlier ratio; try a stricter ratio threshold or a better image pair"

    # 4. Plausible alignment fallback
    return "alignment probably usable; inspect the seam, exposure differences, and parallax"

```

## Building the Master Stitch Function

Now that your diagnose_stitch helper can read a report and explain a stitch, it's time to build the engine that fills that report in the first place.

In this exercise, you will complete the body of stitch_pair, the master function that runs every stitching tool in order. The report dictionary is already set up for you, starting at "failed", so your job is to walk through the pipeline and update it as each step succeeds.

Follow the TODO comments from top to bottom. Along the way, you will:

    Preprocess both images and detect their keypoints and descriptors.
    Match the descriptors and record the keypoint and match counts in the report.
    Add the guard for "fewer than four matches" so the function stops early instead of crashing.
    Estimate the homography inside a try/except ValueError block, then save the inlier count and ratio.
    Add the guard for "fewer than four inliers", then warp, composite, and crop the final image.
    Set the status to "ok", call diagnose_stitch(report), and return the result with its report.

Take it one step at a time, and you'll have a complete stitcher that explains itself every step of the way.

```
import cv2
import numpy as np

from cvkit import preprocess_for_features
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography, warp_to_canvas


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))

    if coords is None:
        return image

    x, y, w, h = cv2.boundingRect(coords)
    return image[y : y + h, x : x + w]


def diagnose_stitch(report):
    if report["matches"] < 20:
        return "too few descriptor matches; choose images with more overlap or texture"

    if report["inliers"] < 10:
        return "too few geometric inliers; inspect the inlier view"

    if report["inlier_ratio"] < 0.25:
        return "low inlier ratio; try a stricter ratio threshold or a better image pair"

    return "alignment probably usable; inspect the seam, exposure differences, and parallax"


def stitch_pair(left, right, method="sift", ratio=0.75, ransac_threshold=5.0):
    report = {
        "status": "failed",
        "homography_direction": "left/source -> right/target",
        "method": method,
        "ratio": ratio,
        "ransac_threshold": ransac_threshold,
        "keypoints_left": 0,
        "keypoints_right": 0,
        "matches": 0,
        "inliers": 0,
        "inlier_ratio": 0.0,
        "diagnosis": "",
    }

    # TODO: Preprocess the left and right images for feature detection.
    # Use preprocess_for_features on each image to get grayscale versions.

    # TODO: Detect keypoints and compute descriptors for each grayscale image.
    # Use detect_and_compute and pass method=method.

    # TODO: Match the descriptors between the two images.
    # Use match_descriptors and pass ratio=ratio.

    # TODO: Update the report. Set "keypoints_left", "keypoints_right", and
    # "matches" to the counts you found (use len(...)).

    # TODO: If there are fewer than four matches, set this diagnosis:
    # "fewer than four matches; homography cannot be estimated"
    # then return None, report

    # TODO: Estimate the homography inside a try/except ValueError block.
    # In the try, call estimate_homography(kp1, kp2, matches,
    # ransac_threshold=ransac_threshold) to get homography and inliers.
    # In the except, catch the error as exc, set the diagnosis to
    # f"homography failed: {exc}", then return None, report

    # TODO: Update the report. Set "inliers" to int(inliers.sum()) and
    # "inlier_ratio" to float(inliers.mean()) if len(inliers) else 0.0

    # TODO: If there are fewer than four inliers, set this diagnosis:
    # "fewer than four inliers; alignment is not reliable"
    # then return None, report

    # TODO: Warp the images with warp_to_canvas, then build the final image
    # with crop_black(composite(...)).

    # TODO: Set "status" to "ok", set "diagnosis" using diagnose_stitch(report),
    # then return result, report

    return None, report
```

Here is the completed implementation for `stitching.py`:

```python
import cv2
import numpy as np

from cvkit import preprocess_for_features
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography, warp_to_canvas


def composite(warped_source, target_canvas):
    source_mask = np.any(warped_source > 0, axis=2)
    target_mask = np.any(target_canvas > 0, axis=2)

    result = target_canvas.copy()

    source_only = source_mask & ~target_mask
    overlap = source_mask & target_mask

    result[source_only] = warped_source[source_only]
    result[overlap] = (
        0.5 * warped_source[overlap] + 0.5 * target_canvas[overlap]
    ).astype(np.uint8)

    return result


def crop_black(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero((gray > 0).astype(np.uint8))

    if coords is None:
        return image

    x, y, w, h = cv2.boundingRect(coords)
    return image[y : y + h, x : x + w]


def diagnose_stitch(report):
    if report["matches"] < 20:
        return "too few descriptor matches; choose images with more overlap or texture"

    if report["inliers"] < 10:
        return "too few geometric inliers; inspect the inlier view"

    if report["inlier_ratio"] < 0.25:
        return "low inlier ratio; try a stricter ratio threshold or a better image pair"

    return "alignment probably usable; inspect the seam, exposure differences, and parallax"


def stitch_pair(left, right, method="sift", ratio=0.75, ransac_threshold=5.0):
    report = {
        "status": "failed",
        "homography_direction": "left/source -> right/target",
        "method": method,
        "ratio": ratio,
        "ransac_threshold": ransac_threshold,
        "keypoints_left": 0,
        "keypoints_right": 0,
        "matches": 0,
        "inliers": 0,
        "inlier_ratio": 0.0,
        "diagnosis": "",
    }

    # 1. Preprocess images to grayscale
    left_gray = preprocess_for_features(left)
    right_gray = preprocess_for_features(right)

    # 2. Detect keypoints and compute descriptors
    kp1, des1 = detect_and_compute(left_gray, method=method)
    kp2, des2 = detect_and_compute(right_gray, method=method)

    # 3. Match descriptors
    matches = match_descriptors(des1, des2, ratio=ratio)

    # 4. Update keypoint and match counts
    report["keypoints_left"] = len(kp1)
    report["keypoints_right"] = len(kp2)
    report["matches"] = len(matches)

    # 5. Guard against fewer than 4 matches
    if len(matches) < 4:
        report["diagnosis"] = "fewer than four matches; homography cannot be estimated"
        return None, report

    # 6. Estimate homography with error boundary
    try:
        homography, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=ransac_threshold,
        )
    except ValueError as exc:
        report["diagnosis"] = f"homography failed: {exc}"
        return None, report

    # 7. Update inlier statistics
    report["inliers"] = int(inliers.sum())
    report["inlier_ratio"] = float(inliers.mean()) if len(inliers) else 0.0

    # 8. Guard against fewer than 4 inliers
    if report["inliers"] < 4:
        report["diagnosis"] = "fewer than four inliers; alignment is not reliable"
        return None, report

    # 9. Warp, composite, and crop final image
    warped, target_canvas = warp_to_canvas(left, right, homography)
    result = crop_black(composite(warped, target_canvas))

    # 10. Update status and final diagnosis
    report["status"] = "ok"
    report["diagnosis"] = diagnose_stitch(report)

    return result, report

```

## Wiring the Stitcher into Action

Nice work building stitch_pair last time — now it's time to put that function to use. In this exercise, you will finish the command-line script in solution.py so that a user can stitch two images directly from the terminal.

The main() function already parses the arguments and reads both images for you. Your job is to complete the rest of the pipeline by following the TODO comments.

Inside main(), complete these steps:

    Call stitch_pair with left, right, and the parsed settings (method, ratio, and ransac_threshold), then capture the result and report.
    Print the report with print_report(report).
    If result is None, terminate the program with raise SystemExit("stitching failed; see diagnosis above").
    Otherwise, save the image with cv2.imwrite(args.out, result), then display it using cv2.imshow, cv2.waitKey(0), and cv2.destroyAllWindows.

You can execute the program in the terminal, like python solution.py sample_images/building/1.jpg sample_images/building/2.jpg sample_images/building/3.jpg and see the output in the Image Server! Wiring this up turns all your hard work into a real tool that anyone can run on their own photos.

```
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


def print_report(report):
    for key, value in report.items():
        print(f"{key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--out", default="stitched_pair.jpg")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    # TODO: Call stitch_pair with left, right, and the parsed settings
    # (method=args.method, ratio=args.ratio,
    # ransac_threshold=args.ransac_threshold). Capture result, report.

    # TODO: Print the report using print_report(report).

    # TODO: If result is None, stop the program with:
    # raise SystemExit("stitching failed; see diagnosis above")

    # TODO: Save the image with cv2.imwrite(args.out, result), then display it
    # with cv2.imshow("stitched pair", result), cv2.waitKey(0), and
    # cv2.destroyAllWindows().


if __name__ == "__main__":
    main()
```

Here is the completed implementation for `solution.py`:

```python
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


def print_report(report):
    for key, value in report.items():
        print(f"{key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--out", default="stitched_pair.jpg")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    # 1. Run stitch_pair with CLI arguments
    result, report = stitch_pair(
        left,
        right,
        method=args.method,
        ratio=args.ratio,
        ransac_threshold=args.ransac_threshold,
    )

    # 2. Print diagnostic report
    print_report(report)

    # 3. Guard against failed stitching
    if result is None:
        raise SystemExit("stitching failed; see diagnosis above")

    # 4. Save and display the stitched panorama
    cv2.imwrite(args.out, result)
    cv2.imshow("stitched pair", result)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```